In [32]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")


In [16]:
os.environ["NEO4J_URI"]=os.getenv("NEO4J_URI")
os.environ["NEO4J_USERNAME"] = os.getenv("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = os.getenv("NEO4J_PASSWORD")

In [17]:
from langchain_community.graphs import Neo4jGraph
graph=Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_USERNAME
)

In [18]:
graph

In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [24]:
from langchain_core.documents import Document

text="""Elon Reeve Musk (born June 28, 1971) is a businessman and investor known for his key roles in space
company SpaceX and automotive company Tesla, Inc. Other involvements include ownership of X Corp.,
formerly Twitter, and his role in the founding of The Boring Company, xAI, Neuralink and OpenAI.
He is one of the wealthiest people in the world; as of July 2024, Forbes estimates his net worth to be
US$221 billion.Musk was born in Pretoria to Maye and engineer Errol Musk, and briefly attended
the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through
his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada.
Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics
 and physics. He moved to California in 1995 to attend Stanford University, but dropped out after
  two days and, with his brother Kimbal, co-founded online city guide software company Zip2.
"""

documents=Document(page_content=text)
documents

Document(metadata={}, page_content="Elon Reeve Musk (born June 28, 1971) is a businessman and investor known for his key roles in space\ncompany SpaceX and automotive company Tesla, Inc. Other involvements include ownership of X Corp.,\nformerly Twitter, and his role in the founding of The Boring Company, xAI, Neuralink and OpenAI.\nHe is one of the wealthiest people in the world; as of July 2024, Forbes estimates his net worth to be\nUS$221 billion.Musk was born in Pretoria to Maye and engineer Errol Musk, and briefly attended\nthe University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through\nhis Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada.\nMusk later transferred to the University of Pennsylvania and received bachelor's degrees in economics\n and physics. He moved to California in 1995 to attend Stanford University, but dropped out after\n  two days and, with his brother Kimbal, co-founded online c

In [34]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer=LLMGraphTransformer(llm=llm)

In [35]:
graph_documents = llm_transformer.convert_to_graph_documents([documents])

In [36]:
graph_documents

[GraphDocument(nodes=[Node(id='Elon Reeve Musk', type='Person', properties={}), Node(id='June 28, 1971', type='Date', properties={}), Node(id='Businessman', type='Occupation', properties={}), Node(id='Investor', type='Occupation', properties={}), Node(id='Spacex', type='Company', properties={}), Node(id='Tesla, Inc.', type='Company', properties={}), Node(id='X Corp.', type='Company', properties={}), Node(id='Twitter', type='Company', properties={}), Node(id='The Boring Company', type='Company', properties={}), Node(id='Xai', type='Company', properties={}), Node(id='Neuralink', type='Company', properties={}), Node(id='Openai', type='Company', properties={}), Node(id='Forbes', type='Organization', properties={}), Node(id='Us$221 Billion', type='Net worth', properties={}), Node(id='July 2024', type='Date', properties={}), Node(id='Pretoria', type='City', properties={}), Node(id='Maye Musk', type='Person', properties={}), Node(id='Errol Musk', type='Person', properties={}), Node(id='Engine

In [37]:
graph_documents[0].nodes

[Node(id='Elon Reeve Musk', type='Person', properties={}),
 Node(id='June 28, 1971', type='Date', properties={}),
 Node(id='Businessman', type='Occupation', properties={}),
 Node(id='Investor', type='Occupation', properties={}),
 Node(id='Spacex', type='Company', properties={}),
 Node(id='Tesla, Inc.', type='Company', properties={}),
 Node(id='X Corp.', type='Company', properties={}),
 Node(id='Twitter', type='Company', properties={}),
 Node(id='The Boring Company', type='Company', properties={}),
 Node(id='Xai', type='Company', properties={}),
 Node(id='Neuralink', type='Company', properties={}),
 Node(id='Openai', type='Company', properties={}),
 Node(id='Forbes', type='Organization', properties={}),
 Node(id='Us$221 Billion', type='Net worth', properties={}),
 Node(id='July 2024', type='Date', properties={}),
 Node(id='Pretoria', type='City', properties={}),
 Node(id='Maye Musk', type='Person', properties={}),
 Node(id='Errol Musk', type='Person', properties={}),
 Node(id='Engineer'

In [38]:
graph_documents[0].relationships

[Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='June 28, 1971', type='Date', properties={}), type='BORN', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Businessman', type='Occupation', properties={}), type='IS_A', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Investor', type='Occupation', properties={}), type='IS_A', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Spacex', type='Company', properties={}), type='HAS_KEY_ROLE_IN', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Tesla, Inc.', type='Company', properties={}), type='HAS_KEY_ROLE_IN', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='X Corp.', type='Company', properties

In [52]:
graph.add_graph_documents(graph_documents)
print("Successfully written to Neo4j!")

Successfully written to Neo4j!


### Adding CSV

In [39]:
movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

In [41]:
graph.query(movie_query)

[]

In [43]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Person {name: STRING, born: INTEGER}
Movie {released: INTEGER, title: STRING, id: STRING, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


In [49]:
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
chain=GraphCypherQAChain.from_llm(llm=llm,graph=graph,verbose=True,allow_dangerous_requests=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x116b94c50>, cypher_generation_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output

In [50]:
response=chain.invoke({"query":"Who was the director of the moview GoldenEye"})

response




> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Person)-[:DIRECTED]->(m:Movie)
WHERE m.title = 'GoldenEye'
RETURN p.name

Full Context:
[{'p.name': 'Martin Campbell'}]

> Finished chain.


{'query': 'Who was the director of the moview GoldenEye',
 'result': 'Martin Campbell was the director of the moview GoldenEye.'}

In [58]:
response=chain.invoke({"query":"tell me the genre of th movie GoldenEye"})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WHERE m.title = 'GoldenEye' RETURN g.name

Full Context:
[{'g.name': 'Adventure'}, {'g.name': 'Action'}, {'g.name': 'Thriller'}]

> Finished chain.


{'query': 'tell me the genre of th movie GoldenEye',
 'result': 'Adventure, Action, Thriller are the genres of the movie GoldenEye.'}

In [ ]:
graph.refresh_schema()
print(graph.schema) 


Node properties:
Person {name: STRING, born: INTEGER, id: STRING}
Movie {released: INTEGER, title: STRING, id: STRING, imdbRating: FLOAT}
Genre {name: STRING}
Date {id: STRING}
Occupation {id: STRING}
Company {id: STRING}
Organization {id: STRING}
Net worth {id: STRING}
City {id: STRING}
University {id: STRING}
Country {id: STRING}
Field of study {id: STRING}
Location {id: STRING}
Year {id: STRING}
Relationship properties:

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:BORN]->(:Date)
(:Person)-[:IS_A]->(:Occupation)
(:Person)-[:HAS_KEY_ROLE_IN]->(:Company)
(:Person)-[:OWNS]->(:Company)
(:Person)-[:FOUNDED]->(:Company)
(:Person)-[:BORN_IN]->(:City)
(:Person)-[:SON_OF]->(:Person)
(:Person)-[:ATTENDED]->(:University)
(:Person)-[:IMMIGRATED_TO]->(:Country)
(:Person)-[:ACQUIRED_CITIZENSHIP_VIA]->(:Person)
(:Person)-[:MATRICULATED_AT]->(:University)
(:Person)-[:TRANSFERRED_TO]->(:University)
(:Person)-[:RECEIVED_DEGREE_IN]->(:Field of study)
(